# Avocado Price & Demand Prescriptive Optimization Engine
### Predictive Econometric Demand Elasticity & Non-Linear Prescriptive Profit Optimization (SLSQP)

This notebook demonstrates the end-to-end pipeline combining:
1. **Stage 1 (Predictive Analytics):** Constant-elasticity log-log regression estimating consumer price elasticity ($\beta$) with harmonic seasonality decomposition on Hass Avocado Board data.
2. **Stage 2 (Prescriptive Optimization):** Non-linear profit maximization program (SciPy SLSQP / bounded scalar) subject to supply constraints and inventory bounds, achieving **+33.45% net profit uplift**.

In [1]:
import pandas as pd
import numpy as np
import os
from scipy.optimize import minimize_scalar

# 1. Load Hass Avocado Board Retail Transactions
data_path = os.path.join("data", "avocado.csv")
if not os.path.exists(data_path):
    data_path = "avocado.csv"

df_raw = pd.read_csv(data_path)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw['Month'] = df_raw['Date'].dt.month

# Filter for conventional avocados in California
df = df_raw[(df_raw['type'] == 'conventional') & (df_raw['region'] == 'California')].sort_values('Date').reset_index(drop=True)

print(f"Total Raw Dataset Rows: {len(df_raw):,} records across {df_raw['region'].nunique()} US retail regions")
print(f"Filtered California Market Observations: {len(df)} weekly periods")
print(f"Mean Historical Price: ${df['AveragePrice'].mean():.2f} | Mean Weekly Volume: {df['Total_Volume'].mean():,.0f} units")

Total Raw Dataset Rows: 6,580 records across 7 US retail regions
Filtered California Market Observations: 470 weekly periods
Mean Historical Price: $1.15 | Mean Weekly Volume: 2,540,987 units


## 2. Stage 1: Econometric Demand Elasticity Estimation (OLS Log-Log Model)

$$\ln(Q_t) = \alpha + \beta \ln(P_t) + \gamma_1 \sin\left(\frac{2\pi m_t}{12}\right) + \gamma_2 \cos\left(\frac{2\pi m_t}{12}\right) + \epsilon_t$$

In [3]:
# Construct Design Matrix
log_p = np.log(df['AveragePrice'].values)
log_q = np.log(df['Total_Volume'].values)
sin_month = np.sin(2 * np.pi * df['Month'].values / 12.0)
cos_month = np.cos(2 * np.pi * df['Month'].values / 12.0)

n = len(df)
X_design = np.column_stack([np.ones(n), log_p, sin_month, cos_month])

# OLS Estimation via Normal Equations: theta = (X^T X)^(-1) X^T y
theta = np.linalg.solve(X_design.T @ X_design, X_design.T @ log_q)
alpha = theta[0]
beta = theta[1]  # Price Elasticity
gamma_sin = theta[2]
gamma_cos = theta[3]

y_pred = X_design @ theta
r2 = 1.0 - (np.sum((log_q - y_pred)**2) / np.sum((log_q - np.mean(log_q))**2))

print("=" * 75)
print(f"Estimated Price Elasticity (Beta) : {beta:.4f} (Demand is Elastic: |Beta| > 1)")
print(f"Intercept Alpha                   : {alpha:.4f}")
print(f"Model Goodness-of-Fit (R^2)       : {r2:.4f}")
print("=" * 75)

Estimated Price Elasticity (Beta) : -1.2258 (Demand is Elastic: |Beta| > 1)
Intercept Alpha                   : 14.9003
Model Goodness-of-Fit (R^2)       : 0.7530


## 3. Stage 2: Prescriptive Non-Linear Profit Optimization (SciPy SLSQP)

$$\max_{P} \quad \Pi(P) = (P - c) \cdot \hat{Q}(P)$$
$$\text{s.t.} \quad \hat{Q}(P) \le \text{CapacityLimit}, \quad P \in [P_{\min}, P_{\max}]$$

In [5]:
unit_cost = 0.60  # $0.60 per avocado unit procurement and handling cost
max_supply = 3000000.0  # 3M avocados maximum weekly supply capacity
eval_month = 6  # Summer peak month

def predict_demand(price, month=eval_month):
    sin_m = np.sin(2 * np.pi * month / 12.0)
    cos_m = np.cos(2 * np.pi * month / 12.0)
    return np.exp(alpha + beta * np.log(price) + gamma_sin * sin_m + gamma_cos * cos_m)

def neg_profit(p):
    q = predict_demand(p)
    if q > max_supply:
        return -((p - unit_cost) * max_supply - 1e6 * (q - max_supply))
    return -((p - unit_cost) * q)

# Prescriptive optimization across allowable price band [$0.70, $2.50]
res = minimize_scalar(neg_profit, bounds=(0.70, 2.50), method='bounded')
opt_price = float(res.x)
opt_demand = predict_demand(opt_price)
opt_revenue = opt_price * opt_demand
opt_profit = (opt_price - unit_cost) * opt_demand

# Baseline comparison
hist_price = float(df['AveragePrice'].mean())
hist_demand = predict_demand(hist_price)
hist_profit = (hist_price - unit_cost) * hist_demand
uplift_pct = ((opt_profit - hist_profit) / hist_profit) * 100.0

print("=" * 75)
print(f"Baseline Historical Retail Price : ${hist_price:.2f} per unit (Profit: ${hist_profit:,.2f}/wk)")
print(f"Prescriptive Optimal Retail Price: ${opt_price:.2f} per unit")
print(f"Forecasted Weekly Demand Volume  : {opt_demand:,.0f} units")
print(f"Projected Optimal Weekly Profit  : ${opt_profit:,.2f} per week")
print(f"Net Profit Margin Expansion      : +{uplift_pct:.2f}% uplift over historical pricing")
print("=" * 75)

Baseline Historical Retail Price : $1.15 per unit (Profit: $1,378,967.15/wk)
Prescriptive Optimal Retail Price: $2.50 per unit
Forecasted Weekly Demand Volume  : 968,549 units
Projected Optimal Weekly Profit  : $1,840,238.15 per week
Net Profit Margin Expansion      : +33.45% uplift over historical pricing
